# Michigan Traders — Module 3
# The 5 Pillars of a QuantConnect Algorithm  ·  *interactive workbook*

**Series:** MAT Education · QuantConnect Core
**Level:** Intermediate (builds on Modules 1–2; assumes Python fluency)
**Format:** **Reference + Fill-in-the-Pillar** — every pillar gets an explanation, worked examples, and a `# TODO` exercise for you.

---

Every QuantConnect algorithm — from a two-line moving-average crossover to a production-grade ML strategy — is assembled from the same **five building blocks**. Learn these five pillars and you can read, write, and extend *any* QC algo you encounter this year.

After working through each pillar individually you'll assemble the pieces into two complete, paste-ready algorithms: an MA crossover (Pillars 1–4) and a monthly cross-sectional momentum strategy (all 5 Pillars).

## How to use this notebook

This module has two kinds of cell, and it matters which is which.

| Cell type | Where it runs | What to do |
|---|---|---|
| 🔵 **QC cell** | QuantConnect (LEAN) | Copy into an algorithm project. It will **not** run in a local Jupyter kernel — `QCAlgorithm` only exists inside the LEAN engine. |
| 🟢 **Local cell** | your laptop's Jupyter | Run it. These carry a self-check that prints ✅ when your answer is right. |

Most of this module is 🔵, because the five pillars *are* the QC API. But every pillar also has a 🟢 exercise covering the arithmetic and the reasoning behind it — the parts you can get wrong without the compiler ever telling you. Those are the ones that check themselves.

For each pillar you'll see:

1. **An explanation** of what the pillar does and *why* it exists.
2. **Worked examples** — read the code and comments carefully; the same patterns repeat in every QC algo.
3. **✏️ Your turn (🔵)** — an algorithm stub with `# TODO` blanks. Fill in the missing pillar, then paste it into QC and check it compiles.
4. **✏️ Your turn (🟢)** — a local exercise on the same pillar's logic, with a self-check.

Answers to both kinds are in `03_Five_Pillars_of_a_QuantConnect_Algorithm_SOLUTIONS.ipynb`.

The local cells use `pandas` and `numpy` from Module 2. Work top to bottom; use the cheat sheet at the end as a quick-reference lookup.

## The QuantConnect Lifecycle

One mental model to lock in before the five pillars:

```
  ┌──────────────────────────────────────────────────────────┐
  │                  Your Algorithm Class                    │
  │                                                          │
  │   initialize()         ← called ONCE at startup          │
  │       set dates, cash, assets, indicators                │
  │                                                          │
  │   on_data(data: Slice) ← called EVERY BAR               │
  │       read prices → check signals → place orders         │
  └──────────────────────────────────────────────────────────┘
```

`initialize` is your setup code — it runs once when the backtest begins. `on_data` is your main loop — QuantConnect calls it every time a new bar of data arrives (once per day for daily data, once per minute for minute data, and so on). Everything else — indicators, orders, portfolio checks — lives inside one of these two methods.

### The empty skeleton

Every QC algorithm starts with this two-method shell. The class inherits from `QCAlgorithm`, which provides all the tools you'll need — data feeds, order routing, portfolio tracking, logging, and more — as `self.*` methods.

In [ ]:
class MyAlgorithm(QCAlgorithm):
    # Minimal QC algorithm skeleton.
    # Copy this into any QuantConnect project as your starting point.

    def initialize(self):
        # Pillar 1: tell QC when to run, how much cash, which assets to load
        pass

    def on_data(self, data: Slice):
        # Pillar 2: handle new data
        # Pillar 3: check indicators
        # Pillar 4: place orders
        pass

## Pillar 1 — Initialize

`initialize` runs **once** at the very start of every backtest (or when you go live). It's where you configure everything QC needs before the first bar of data arrives:

- **Date range** — which period to backtest
- **Starting capital** — how much paper money to begin with
- **Asset subscriptions** — which tickers to load, at what data resolution
- **Indicators** — set up moving averages, RSI, etc. (Pillar 3 covers this in depth)
- **Benchmark** — what to compare performance against (default: SPY)

### Setting up dates and cash

Three methods, always called in this order:

| Method | What it does |
|---|---|
| `self.set_start_date(year, month, day)` | First bar of backtest data |
| `self.set_end_date(year, month, day)` | Last bar (omit → today) |
| `self.set_cash(amount)` | Starting portfolio value in USD |

In [ ]:
class Pillar1_DatesAndCash(QCAlgorithm):
    def initialize(self):
        # ── dates ──────────────────────────────────────────────────────────
        self.set_start_date(2022, 1, 1)   # backtest starts 2022-01-01
        self.set_end_date(2024, 12, 31)   # backtest ends   2024-12-31

        # ── capital ────────────────────────────────────────────────────────
        self.set_cash(100_000)            # $100,000 starting cash

        # No assets subscribed yet — this algo does nothing, but it's valid QC code.

### Subscribing to data with add_equity

`add_equity` tells QC which ticker to load price data for. It returns an `Equity` object; we call `.symbol` on it and store the result on `self` so we can reference the same symbol handle throughout the algorithm.

```
self.symbol = self.add_equity(ticker, resolution).symbol
```

| Argument | Example | Notes |
|---|---|---|
| `ticker` | `"SPY"` | US equities and ETFs only in this course |
| `resolution` | `Resolution.DAILY` | `DAILY` / `HOUR` / `MINUTE` (all UPPER_SNAKE_CASE enums) |

In [ ]:
class Pillar1_AddEquity(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)

        # add_equity returns an Equity object; .symbol gives us a Symbol handle.
        # We save it on self so on_data can use it without re-looking up the ticker.
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # Optional: benchmark against QQQ instead of the default SPY
        self.set_benchmark("QQQ")

### ✏️ Your turn — initialize a two-stock strategy

Set up an algorithm that:
- Runs from **2020-01-01** to **2023-12-31**
- Starts with **$50,000** cash
- Loads **AAPL** and **TLT** at daily resolution, storing their Symbol objects as `self.aapl` and `self.tlt`

Write only the `initialize` method body — the rest of the class is already provided.

In [ ]:
# 🔵 QC cell — no local self-check for this one
class TwoStockAlgo(QCAlgorithm):
    def initialize(self):
        # TODO: set_start_date(2020, 1, 1)

        # TODO: set_end_date(2023, 12, 31)

        # TODO: set_cash(50_000)

        # TODO: self.aapl = self.add_equity("AAPL", Resolution.DAILY).symbol
        # TODO: self.tlt  = self.add_equity("TLT",  Resolution.DAILY).symbol

    def on_data(self, data: Slice):   # Pillar 2 — provided for context
        pass

> **Verify:** Your `initialize` should call exactly four methods: `set_start_date`, `set_end_date`, `set_cash`, and `add_equity` (twice). Compare with `03_Five_Pillars_SOLUTIONS.ipynb`, or paste the class into a QC project — it should compile without errors.

### 🟢 The `initialize` contract, checked locally

Everything above is 🔵 — it only runs inside LEAN. But the *decisions* inside `initialize` can be wrong in ways the compiler will never catch. A backtest with the end date before the start date compiles perfectly and reports nothing. One with zero cash compiles and buys nothing.

Strip `initialize` down to the settings it actually establishes and you get a plain dictionary, which you can reason about on your laptop.

In [1]:
import pandas as pd

# Every initialize() reduces to a handful of settings. Writing them as a dict
# lets us check them without the LEAN engine.
config = {
    "start": pd.Timestamp("2020-01-01"),
    "end": pd.Timestamp("2023-01-01"),
    "cash": 100_000,
    "symbols": ["SPY", "TLT"],
    "resolution": "daily",
}

for key, value in config.items():
    print(f"{key:12s} {value}")

span_years = (config["end"] - config["start"]).days / 365.25
print(f"\nbacktest spans {span_years:.1f} years")

start        2020-01-01 00:00:00
end          2023-01-01 00:00:00
cash         100000
symbols      ['SPY', 'TLT']
resolution   daily

backtest spans 3.0 years


### ✏️ Your turn — validate an initialize config (🟢 local)

Write `validate_initialize(config)` returning a **sorted list** of problem names found in a config dict like the one above. Use exactly these names:

| name | reported when |
|---|---|
| `"start_not_before_end"` | `start` is not strictly before `end` |
| `"cash_not_positive"` | `cash` is zero or negative |
| `"no_symbols"` | `symbols` is empty |
| `"bad_resolution"` | `resolution` is not one of `daily`, `hour`, `minute` |

Return `[]` for a valid config. Then set `good_problems` for the `config` above and `bad_problems` for `broken` (defined in the stub).

In [ ]:
broken = {
    "start": pd.Timestamp("2023-01-01"),
    "end": pd.Timestamp("2020-01-01"),
    "cash": 0,
    "symbols": [],
    "resolution": "weekly",
}

VALID_RESOLUTIONS = {"daily", "hour", "minute"}


def validate_initialize(config):
    problems = []
    # TODO: append a name for each rule the config breaks, then sort
    ...


good_problems = validate_initialize(config)
bad_problems = validate_initialize(broken)

print(f"config -> {good_problems}")
print(f"broken -> {bad_problems}")

In [ ]:
assert good_problems == [], f"the worked config is valid, got {good_problems}"
assert bad_problems == ["bad_resolution", "cash_not_positive",
                        "no_symbols", "start_not_before_end"],     f"expected all four problems, SORTED, got {bad_problems}"

# Each rule must fire independently.
import copy
for field, value, expect in [("cash", -5, "cash_not_positive"),
                             ("symbols", [], "no_symbols"),
                             ("resolution", "weekly", "bad_resolution")]:
    c = copy.copy(config)
    c[field] = value
    assert validate_initialize(c) == [expect],         f"setting {field}={value!r} should report exactly ['{expect}']"

# Equal dates are not a valid range - the comparison must be strict.
same = copy.copy(config)
same["end"] = same["start"]
assert validate_initialize(same) == ["start_not_before_end"],     "start == end is not a valid backtest window; use a strict < comparison"
print("✅ Correct!  A config that compiles is not a config that means anything.",
      "These four settings decide what your backtest is even measuring.")

## Pillar 2 — Data: on_data and the Slice

Every time a new bar of market data arrives, QC calls `on_data(self, data: Slice)`. The `Slice` is a snapshot of that bar — prices, volume, and any other data your algorithm subscribed to.

Think of `Slice` as a dictionary keyed by `Symbol`:
- `data[symbol]` → a `TradeBar` object with `.open`, `.close`, `.high`, `.low`, `.volume`
- `data.contains_key(symbol)` → check that data actually arrived (not every symbol trades every bar)

### The guard pattern

Always confirm that the data you need arrived before using it. Missing data is normal — exchanges are closed, trading halts occur, newly added universe members haven't loaded yet.

Two equivalent ways to guard:

```python
if self.symbol not in data:             # Python "in" operator — most readable
    return

if not data.contains_key(self.symbol):  # explicit QC method — same result
    return
```

Both short-circuit `on_data` for that bar gracefully.

### What's in a TradeBar

| Attribute | Type | What it holds |
|---|---|---|
| `bar.open` | `float` | opening price of the bar |
| `bar.high` | `float` | highest price of the bar |
| `bar.low` | `float` | lowest price of the bar |
| `bar.close` | `float` | closing price of the bar |
| `bar.volume` | `float` | total shares traded |
| `bar.time` | `datetime` | timestamp of the bar |

In [ ]:
class Pillar2_ReadPrice(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

    def on_data(self, data: Slice):
        # Guard: skip bars where SPY data didn't arrive
        if self.symbol not in data:
            return

        bar    = data[self.symbol]            # TradeBar for this bar
        close  = bar.close                    # closing price
        volume = bar.volume                   # shares traded that day

        # self.log() writes a line to the QC backtest log panel
        self.log(f"SPY | close={close:.2f}  volume={volume:,.0f}")

### ✏️ Your turn — log AAPL's daily closing price

Complete `on_data` so it:
- Guards against missing AAPL data
- Reads the closing price from the `Slice`
- Logs a message in exactly this format: `AAPL close: 182.45` (two decimal places)

The `initialize` method is already filled in for you.

In [ ]:
# 🔵 QC cell — no local self-check for this one
class Pillar2Exercise(QCAlgorithm):
    def initialize(self):                  # Pillar 1 — already done
        self.set_start_date(2023, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("AAPL", Resolution.DAILY).symbol

    def on_data(self, data: Slice):
        # TODO: guard — return if self.symbol not in data

        # TODO: read the closing price from data[self.symbol].close

        # TODO: log "AAPL close: 182.45" (replace 182.45 with the actual close)
        pass

> **Verify:** Your `on_data` should have three parts: a guard `return`, a `.close` read, and a `self.log(f"...")` call. Compare with the solutions file, then paste into QC — the backtest log should fill with daily AAPL prices.

### 🟢 Guarding against a missing bar, checked locally

The single most common runtime crash in a new QC algorithm is indexing a `Slice` for a symbol that is not in it.

A `Slice` is not guaranteed to contain every symbol you subscribed to. A stock can be halted, can be delisted mid-backtest, or can simply not have listed yet on an early date. Reach for `data["AAPL"]` on one of those days and the algorithm dies — after some hours of backtest have already run.

Model a slice as a dict and the guard becomes obvious.

In [4]:
# A Slice behaves like a dict from symbol -> bar. On any given day some
# subscribed symbols may be missing: a halt, a holiday, or a late listing.
slice_day = {"SPY": {"close": 452.10}, "TLT": {"close": 98.40}}
subscribed = ["SPY", "TLT", "AAPL"]

for symbol in subscribed:
    # In QC this guard is: if not data.bars.contains_key(symbol): continue
    if symbol in slice_day:
        print(f"{symbol:5s} close {slice_day[symbol]['close']:7.2f}")
    else:
        print(f"{symbol:5s} no bar today -> skip, do not index it")

SPY   close  452.10
TLT   close   98.40
AAPL  no bar today -> skip, do not index it


### ✏️ Your turn — handle an incomplete Slice (🟢 local)

Write `closes_available(slice_day, subscribed)` returning a dict of `symbol -> close` containing **only** the subscribed symbols present in the slice.

Then write `complete_days(days, subscribed)` returning a list of the **indices** of the days on which every subscribed symbol had a bar.

Apply them to `week` (in the stub) as `day0_closes` and `full_days`.

In [ ]:
week = [
    {"SPY": {"close": 452.10}, "TLT": {"close": 98.40}},                          # AAPL halted
    {"SPY": {"close": 455.00}, "TLT": {"close": 98.10}, "AAPL": {"close": 189.2}},
    {"SPY": {"close": 451.75}, "AAPL": {"close": 187.9}},                         # TLT missing
    {"SPY": {"close": 458.30}, "TLT": {"close": 97.80}, "AAPL": {"close": 191.4}},
]
subscribed = ["SPY", "TLT", "AAPL"]


def closes_available(slice_day, subscribed):
    # TODO: only the symbols that actually have a bar
    ...


def complete_days(days, subscribed):
    # TODO: indices of days where every subscribed symbol is present
    ...


day0_closes = closes_available(week[0], subscribed)
full_days = complete_days(week, subscribed)

print(f"day 0 -> {day0_closes}")
print(f"complete days -> {full_days}")

In [ ]:
assert day0_closes == {"SPY": 452.10, "TLT": 98.40},     f"day 0 has no AAPL bar, so it must not appear, got {day0_closes}"
assert "AAPL" not in day0_closes, "never invent a price for a symbol with no bar"
assert full_days == [1, 3], f"only days 1 and 3 have all three symbols, got {full_days}"

# An empty slice must return an empty dict, not raise.
assert closes_available({}, subscribed) == {}, "an empty Slice is legal; handle it"
# A symbol in the slice but not subscribed must be ignored.
extra = {"SPY": {"close": 1.0}, "GLD": {"close": 2.0}}
assert closes_available(extra, subscribed) == {"SPY": 1.0},     "only report symbols you actually subscribed to"
assert complete_days([], subscribed) == [], "no days in, no days out"
print(f"✅ Correct!  {len(full_days)} of {len(week)} days were fully populated.",
      "Guard every Slice access - a missing bar is normal, and an unguarded",
      "index kills the whole backtest hours in.")

## Pillar 3 — Indicators

Indicators transform raw price data into trading signals — a moving average smooths price noise, RSI measures momentum, Bollinger Bands quantify volatility. QuantConnect ships 100+ built-in indicators.

**Two ways to create an indicator:**

1. **Helper method (recommended):** `self.sma(symbol, period, resolution)` — QC automatically wires up the data feed.
2. **Manual:** `SimpleMovingAverage(period)` + `self.register_indicator(symbol, indicator, resolution)` — more control, same end result.

We use helper methods throughout this course.

### Creating indicators in initialize

Indicators are created in `initialize` (not in `on_data`) and stored on `self` so every call to `on_data` can access them. Common helpers:

| Helper | What it creates |
|---|---|
| `self.sma(symbol, period, resolution)` | Simple Moving Average |
| `self.ema(symbol, period, resolution)` | Exponential Moving Average |
| `self.rsi(symbol, period, resolution)` | Relative Strength Index (0–100) |
| `self.bb(symbol, period, k, resolution)` | Bollinger Bands |
| `self.macd(symbol, fast, slow, signal, resolution)` | MACD |

In [ ]:
class Pillar3_CreateIndicators(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # Create a 20-day and a 50-day simple moving average.
        # QC automatically feeds SPY's daily closing prices into both indicators.
        self.sma_20 = self.sma(self.symbol, 20, Resolution.DAILY)
        self.sma_50 = self.sma(self.symbol, 50, Resolution.DAILY)

        # Warm up: pre-load 50 bars of history so both indicators are ready
        # on the very first on_data call.
        self.set_warm_up(50)

### Reading indicator values in on_data

An indicator has two key attributes:

| Attribute | Type | Meaning |
|---|---|---|
| `.is_ready` | `bool` | `True` once enough bars have been processed (e.g. 20 bars for SMA20) |
| `.current.value` | `float` | The most recently computed value |

**Always check `.is_ready` before using `.current.value`.** An unready indicator returns 0, which would produce garbage signals on the first few bars.

In [ ]:
class Pillar3_ReadIndicators(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol
        self.sma_20 = self.sma(self.symbol, 20, Resolution.DAILY)
        self.sma_50 = self.sma(self.symbol, 50, Resolution.DAILY)
        self.set_warm_up(50)

    def on_data(self, data: Slice):
        if self.symbol not in data:
            return

        # Guard: wait until both indicators have processed enough history
        if not self.sma_20.is_ready or not self.sma_50.is_ready:
            return

        fast  = self.sma_20.current.value   # today's 20-day SMA value
        slow  = self.sma_50.current.value   # today's 50-day SMA value
        price = data[self.symbol].close

        self.log(f"SPY={price:.2f}  SMA20={fast:.2f}  SMA50={slow:.2f}  "
                 f"signal={'BULL' if fast > slow else 'BEAR'}")

In [7]:
# Why set_warm_up matters
#
# set_warm_up(n) pre-loads n bars of historical data before the backtest
# date range begins, so indicators are already computed on day 1.
#
# Without warm_up: on_data fires from day 1 but .is_ready stays False for
# the first (period - 1) bars -- you'd silently skip early signals.
#
# Rule of thumb: set_warm_up to the longest indicator period you use.
#   Two indicators with periods 20 and 50 -> set_warm_up(50).

### ✏️ Your turn — add an RSI indicator

Complete the algorithm below. In `initialize`, add an **RSI with period 14**:
```python
self.rsi_14 = self.rsi(self.symbol, 14, Resolution.DAILY)
```
Then in `on_data`:
- Guard against missing data AND an unready RSI
- Read the RSI value and log: `RSI=67.3` (one decimal place)

In [ ]:
# 🔵 QC cell — no local self-check for this one
class Pillar3Exercise(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # TODO: create self.rsi_14 = self.rsi(self.symbol, 14, Resolution.DAILY)

        # TODO: call self.set_warm_up(14)

    def on_data(self, data: Slice):
        # TODO: guard — return if self.symbol not in data

        # TODO: guard — return if not self.rsi_14.is_ready

        # TODO: read self.rsi_14.current.value and log "RSI=67.3"
        pass

> **Verify:** Your `initialize` should call `self.rsi(...)` and `self.set_warm_up(14)`. Your `on_data` should have two guards and one `self.log(...)`. Compare with the solutions file.

### 🟢 Warm-up arithmetic, checked locally

`set_warm_up` is the setting people leave out, and leaving it out does not raise an error. It silently produces a backtest whose first weeks traded on indicators reading zero.

The arithmetic is small enough to do exactly. An indicator of period *N* is not ready until it has seen *N* bars, so the warm-up you need is the **longest** period in the algorithm — get that number wrong by one and your slowest indicator is still cold on day one.

In [8]:
# An indicator of period N is not ready until N bars have arrived.
periods = {"sma_fast": 20, "sma_slow": 50, "rsi": 14}

for name, p in sorted(periods.items(), key=lambda kv: kv[1]):
    print(f"{name:9s} period {p:3d} -> ready once {p} bars have arrived")

print(f"\nset_warm_up({max(periods.values())})   # the longest period governs")
print("after 20 bars, ready:", sorted(n for n, p in periods.items() if 20 >= p))

rsi       period  14 -> ready once 14 bars have arrived
sma_fast  period  20 -> ready once 20 bars have arrived
sma_slow  period  50 -> ready once 50 bars have arrived

set_warm_up(50)   # the longest period governs
after 20 bars, ready: ['rsi', 'sma_fast']


### ✏️ Your turn — compute the warm-up you need (🟢 local)

Write `required_warmup(periods)` returning the number of bars to warm up for a dict of `name -> period`. An empty dict needs `0`.

Then write `ready_after(periods, bars_seen)` returning the **sorted list of names** ready once `bars_seen` bars have arrived. An indicator of period *N* is ready when `bars_seen >= N` — the boundary is inclusive.

Set `warmup` to `required_warmup(my_periods)`, and `ready_at_20` / `ready_at_50` to `ready_after(my_periods, 20)` and `ready_after(my_periods, 50)`.

In [ ]:
my_periods = {"sma_fast": 20, "sma_slow": 50, "rsi": 14, "atr": 20}


def required_warmup(periods):
    # TODO: how many bars before the slowest indicator is ready?
    ...


def ready_after(periods, bars_seen):
    # TODO: which names are ready at this bar count?
    ...


warmup = required_warmup(my_periods)
ready_at_20 = ready_after(my_periods, 20)
ready_at_50 = ready_after(my_periods, 50)

print(f"set_warm_up({warmup})")
print(f"ready after 20 bars: {ready_at_20}")
print(f"ready after 50 bars: {ready_at_50}")

In [ ]:
assert warmup == 50, f"the slowest indicator has period 50, got {warmup}"
assert required_warmup({}) == 0, "an algorithm with no indicators needs no warm-up"
assert required_warmup({"only": 7}) == 7, "one indicator: warm up its own period"

assert ready_at_20 == ["atr", "rsi", "sma_fast"],     f"at 20 bars the period-50 indicator is still cold, got {ready_at_20}"
assert ready_at_50 == ["atr", "rsi", "sma_fast", "sma_slow"],     f"at 50 bars everything is ready, got {ready_at_50}"

# The boundary is inclusive: period 20 IS ready on bar 20, not bar 21.
assert "sma_fast" in ready_after(my_periods, 20),     "a period-20 indicator is ready once 20 bars have arrived; use >= not >"
assert "sma_fast" not in ready_after(my_periods, 19),     "a period-20 indicator is NOT ready at 19 bars"
assert ready_after(my_periods, 0) == [], "nothing is ready before any data arrives"
print(f"✅ Correct!  set_warm_up({warmup}) covers every indicator.",
      "Omit it and the algorithm still runs - it just trades the first 50 bars blind.")

## Pillar 4 — Portfolio & Orders

With a signal in hand (Pillar 3), you need to act on it. Pillar 4 covers:
- **Checking positions** — are we already in this trade?
- **Placing orders** — enter and exit positions
- **Reading portfolio state** — total value, cash on hand

### Checking portfolio state

| Expression | Returns | What it means |
|---|---|---|
| `self.portfolio[symbol].invested` | `bool` | `True` if we hold any shares |
| `self.portfolio[symbol].quantity` | `float` | Shares held (negative = short) |
| `self.portfolio.total_portfolio_value` | `float` | Cash + all open positions (USD) |
| `self.portfolio.cash` | `float` | Uninvested cash |

In [ ]:
class Pillar4_CheckPortfolio(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

    def on_data(self, data: Slice):
        if self.symbol not in data:
            return

        invested = self.portfolio[self.symbol].invested    # bool
        qty      = self.portfolio[self.symbol].quantity    # shares
        total    = self.portfolio.total_portfolio_value    # USD
        cash     = self.portfolio.cash                     # USD

        self.log(f"invested={invested}  qty={qty}  total=${total:,.0f}  cash=${cash:,.0f}")

### Placing orders

Three methods cover 90% of use cases:

| Method | What it does |
|---|---|
| `self.set_holdings(symbol, weight)` | Target `weight` fraction of portfolio in `symbol` (0 = flat, 1.0 = 100% long, -1 = 100% short) |
| `self.liquidate(symbol)` | Close the entire position in `symbol` immediately |
| `self.market_order(symbol, quantity)` | Buy/sell exactly `quantity` shares at market price |

**`set_holdings` is the most common** for single-asset and multi-asset strategies because it handles position sizing automatically based on current portfolio value.

In [ ]:
class Pillar4_PlaceOrders(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol  = self.add_equity("SPY", Resolution.DAILY).symbol
        self.sma_20  = self.sma(self.symbol, 20, Resolution.DAILY)
        self.set_warm_up(20)

    def on_data(self, data: Slice):
        if self.symbol not in data or not self.sma_20.is_ready:
            return

        price    = data[self.symbol].close
        sma      = self.sma_20.current.value
        invested = self.portfolio[self.symbol].invested

        # Buy signal: price above 20-day average -> go 100% long
        if price > sma and not invested:
            self.set_holdings(self.symbol, 1.0)
            self.log(f"BUY  SPY @ {price:.2f}  (SMA20={sma:.2f})")

        # Sell signal: price fell below the average -> exit
        elif price < sma and invested:
            self.liquidate(self.symbol)
            self.log(f"SELL SPY @ {price:.2f}  (SMA20={sma:.2f})")

### ✏️ Your turn — RSI mean-reversion trade logic

Complete `on_data` below so it:
- **Buys** SPY (100% allocation) when RSI < 30 AND not already long
- **Sells** (liquidates) when RSI > 70 AND currently long
- Logs each trade in the format: `BUY SPY RSI=28.4` or `SELL SPY RSI=71.2`

In [ ]:
# 🔵 QC cell — no local self-check for this one
class Pillar4Exercise(QCAlgorithm):
    def initialize(self):                  # Pillars 1–3 — already done
        self.set_start_date(2022, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol  = self.add_equity("SPY", Resolution.DAILY).symbol
        self.rsi_14  = self.rsi(self.symbol, 14, Resolution.DAILY)
        self.set_warm_up(14)

    def on_data(self, data: Slice):
        if self.symbol not in data or not self.rsi_14.is_ready:
            return

        rsi_val  = self.rsi_14.current.value
        invested = self.portfolio[self.symbol].invested

        # TODO: buy (set_holdings 100%) when rsi_val < 30 and not invested
        #       log "BUY SPY RSI=28.4" with actual RSI value

        # TODO: sell (liquidate) when rsi_val > 70 and invested
        #       log "SELL SPY RSI=71.2" with actual RSI value
        pass

> **Verify:** Your `on_data` should have two `if/elif` branches — one calling `set_holdings`, one calling `liquidate`. Compare with the solutions file, then paste into QC and run a backtest to see trades in the log.

### 🟢 What `set_holdings` actually orders, checked locally

`self.set_holdings("SPY", 0.4)` reads like "buy 40% of SPY". It means something more specific, and the difference matters:

1. The target is **40% of total portfolio value**, not of your cash and not of a fixed number.
2. It is a **target, not an order**. If you already hold 30%, it buys the missing 10%. If you hold 50%, it *sells*.
3. Share counts are whole numbers, so the target value is divided by price and **rounded down**. Rounding to nearest would ask for more money than you have.

That third point is a real bug, not a nicety, and the exercise below checks for it.

In [11]:
portfolio_value = 100_000
price = 452.10
target_pct = 0.40

target_value = portfolio_value * target_pct
target_shares = int(target_value // price)            # floor, never round

print(f"40% of ${portfolio_value:,} = ${target_value:,.2f}")
print(f"at ${price:.2f} -> {target_shares} shares, costing ${target_shares * price:,.2f}")

# It is a target, so the order is the DIFFERENCE from what you hold.
for held in [0, 50, 120]:
    print(f"  holding {held:3d} -> order {target_shares - held:+4d} shares")

40% of $100,000 = $40,000.00
at $452.10 -> 88 shares, costing $39,784.80
  holding   0 -> order  +88 shares
  holding  50 -> order  +38 shares
  holding 120 -> order  -32 shares


### ✏️ Your turn — size a set_holdings order (🟢 local)

Write `target_shares(portfolio_value, price, target_pct)` returning the whole number of shares a `set_holdings` call targets. **Round down** — never ask for more than the target value buys.

Then write `order_quantity(current_shares, target)` returning the signed order needed (positive to buy, negative to sell, `0` to do nothing).

Set `spy_target` for a $100,000 portfolio at $452.10 targeting 40%, and `spy_order` for someone already holding 120 shares.

In [ ]:
def target_shares(portfolio_value, price, target_pct):
    # TODO: target value / price, rounded DOWN to whole shares
    ...


def order_quantity(current_shares, target):
    # TODO: the signed difference
    ...


spy_target = target_shares(100_000, 452.10, 0.40)
spy_order = order_quantity(120, spy_target)

print(f"target {spy_target} shares")
print(f"holding 120 -> order {spy_order:+d}")

In [ ]:
assert spy_target == 88, f"$40,000 / $452.10 = 88.47 -> 88 whole shares, got {spy_target}"
assert spy_order == -32, f"holding 120 with a target of 88 means SELLING 32, got {spy_order}"
assert order_quantity(88, 88) == 0, "already at target means no order at all"
assert target_shares(100_000, 452.10, 0.0) == 0, "a 0% target holds nothing"

# Rounding DOWN is the point: round() here would overspend the portfolio.
overspend = target_shares(10_000, 265.0, 1.0)
assert overspend == 37, (
    f"got {overspend}. $10,000 / $265 = 37.7 shares. Rounding to nearest gives 38, "
    f"which costs ${38 * 265:,} - more than the portfolio has. Round DOWN.")
assert overspend * 265.0 <= 10_000, "the target must be affordable"
print(f"✅ Correct!  set_holdings targets {spy_target} shares, so holding 120 SELLS 32.",
      "It is a destination, not a purchase - and it floors, so it always fits.")

## Pillar 5 — Universe Selection & Scheduling

Pillars 1–4 are sufficient for single-stock strategies. Pillar 5 adds two power tools:

1. **Universe Selection** — instead of hardcoding tickers, let QC dynamically choose which stocks to trade (e.g. top 50 by dollar volume, updated each day).
2. **Scheduling** — trigger a function at a specific recurring time (e.g. rebalance on the first trading day of every month, 30 minutes after the open).

Together, these are what separate a toy algo from a production cross-sectional strategy.

### Universe Selection

`add_universe(filter_fn)` subscribes to a *dynamic* set of securities. QC calls `filter_fn` each day with a list of all tradeable US stocks (`CoarseFundamental` objects) and expects back a list of `Symbol` objects to include.

```python
def coarse_filter(self, coarse):
    # Sort by dollar volume descending, return top 20 symbols
    sorted_stocks = sorted(coarse, key=lambda x: x.dollar_volume, reverse=True)
    return [x.symbol for x in sorted_stocks[:20]]
```

When stocks enter or leave the universe, QC calls `on_securities_changed`:

```python
def on_securities_changed(self, changes):
    for security in changes.added_securities:
        self.log(f"Added: {security.symbol}")
    for security in changes.removed_securities:
        self.liquidate(security.symbol)   # exit position when stock leaves
```

### Scheduling

`self.schedule.on(date_rule, time_rule, function)` calls a function at a recurring time. Common rules:

| Date rule | Fires on... |
|---|---|
| `self.date_rules.every_day()` | Every trading day |
| `self.date_rules.week_start()` | First trading day of each week |
| `self.date_rules.month_start()` | First trading day of each month |

| Time rule | Fires at... |
|---|---|
| `self.time_rules.after_market_open(symbol, n)` | n minutes after open |
| `self.time_rules.before_market_close(symbol, n)` | n minutes before close |
| `self.time_rules.at(hour, minute)` | Specific time (Eastern) |

In [ ]:
class Pillar5_UniverseAndSchedule(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2021, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)

        # SPY used only as a scheduling reference (market-hours anchor)
        self.spy = self.add_equity("SPY", Resolution.DAILY).symbol

        # Universe: top 30 US stocks by dollar volume, refiltered each day by QC
        self.universe_symbols = []
        self.add_universe(self.coarse_filter)

        # Schedule: call self.rebalance on the first trading day of each month,
        # 30 minutes after SPY opens
        self.schedule.on(
            self.date_rules.month_start(),
            self.time_rules.after_market_open(self.spy, 30),
            self.rebalance
        )

    def coarse_filter(self, coarse):
        # Keep only stocks with fundamental data and a price above $5
        eligible = [x for x in coarse if x.has_fundamental_data and x.price > 5]
        return [x.symbol for x in sorted(eligible,
                                         key=lambda x: x.dollar_volume,
                                         reverse=True)[:30]]

    def on_securities_changed(self, changes):
        for security in changes.added_securities:
            if security.symbol not in self.universe_symbols:
                self.universe_symbols.append(security.symbol)
        for security in changes.removed_securities:
            self.liquidate(security.symbol)
            if security.symbol in self.universe_symbols:
                self.universe_symbols.remove(security.symbol)

    def rebalance(self):
        if not self.universe_symbols:
            return
        weight = 1.0 / len(self.universe_symbols)
        for symbol in self.universe_symbols:
            self.set_holdings(symbol, weight)
        self.log(f"Rebalanced: {len(self.universe_symbols)} stocks at {weight:.1%} each")

    def on_data(self, data: Slice):
        pass   # all trading happens in rebalance(), not on_data

### ✏️ Your turn — add a weekly rebalance schedule

The algorithm below has a universe and a `rebalance` function but no scheduler.
Add a `self.schedule.on(...)` call in `initialize` (after `self.add_universe(...)`) so it triggers `self.rebalance`:
- **Date rule:** `self.date_rules.week_start()` — first trading day of each week
- **Time rule:** `self.time_rules.after_market_open(self.spy, 0)` — at market open

The three-line `self.schedule.on(...)` block is all that's needed.

In [ ]:
# 🔵 QC cell — no local self-check for this one
class Pillar5Exercise(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2021, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)
        self.spy = self.add_equity("SPY", Resolution.DAILY).symbol
        self.add_universe(self.coarse_filter)
        self.universe_symbols = []

        # TODO: add self.schedule.on(...) here
        #       date_rule:  self.date_rules.week_start()
        #       time_rule:  self.time_rules.after_market_open(self.spy, 0)
        #       function:   self.rebalance

    def coarse_filter(self, coarse):   # universe — provided
        eligible = [x for x in coarse if x.has_fundamental_data and x.price > 10]
        return [x.symbol for x in sorted(eligible,
                                         key=lambda x: x.dollar_volume,
                                         reverse=True)[:20]]

    def on_securities_changed(self, changes):   # provided
        for s in changes.added_securities:
            if s.symbol not in self.universe_symbols:
                self.universe_symbols.append(s.symbol)
        for s in changes.removed_securities:
            self.liquidate(s.symbol)
            if s.symbol in self.universe_symbols:
                self.universe_symbols.remove(s.symbol)

    def rebalance(self):               # provided
        if not self.universe_symbols:
            return
        weight = 1.0 / len(self.universe_symbols)
        for sym in self.universe_symbols:
            self.set_holdings(sym, weight)
        self.log(f"Rebalanced {len(self.universe_symbols)} stocks")

    def on_data(self, data: Slice):
        pass

> **Verify:** Your `initialize` should contain a `self.schedule.on(...)` call with `date_rules.week_start()`, `time_rules.after_market_open(self.spy, 0)`, and `self.rebalance`. Compare with the solutions file.

### 🟢 When a monthly schedule actually fires, checked locally

`date_rules.month_start()` does not fire on the 1st of the month. It fires on the first day the market is **open**, which in a year is several days that are not the 1st at all.

This matters whenever you reason about your own rebalance dates — counting trades, aligning a signal to a rebalance, or explaining to a reader why a January rebalance is stamped the 2nd. Compute the real dates rather than assuming the calendar.

In [14]:
import pandas as pd

# Business days only - no weekends. (QC also removes market holidays;
# a business-day calendar is close enough to see the effect.)
dates = pd.bdate_range("2023-01-01", "2023-06-30")
by_month = pd.Series(dates).groupby(pd.Series(dates).dt.to_period("M")).first()

for period, first_trading_day in by_month.items():
    calendar_first = period.start_time.date()
    same = "same day" if calendar_first == first_trading_day.date() else "SHIFTED"
    print(f"{period}  calendar 1st {calendar_first}  ->  fires {first_trading_day.date()}  {same}")

2023-01  calendar 1st 2023-01-01  ->  fires 2023-01-02  SHIFTED
2023-02  calendar 1st 2023-02-01  ->  fires 2023-02-01  same day
2023-03  calendar 1st 2023-03-01  ->  fires 2023-03-01  same day
2023-04  calendar 1st 2023-04-01  ->  fires 2023-04-03  SHIFTED
2023-05  calendar 1st 2023-05-01  ->  fires 2023-05-01  same day
2023-06  calendar 1st 2023-06-01  ->  fires 2023-06-01  same day


### ✏️ Your turn — find the real rebalance dates (🟢 local)

Write `month_start_days(dates)` taking a `DatetimeIndex` and returning a **list of `pd.Timestamp`** — the first date present in each calendar month, in order.

Then write `rebalance_count(dates)` returning how many times a `month_start` schedule would fire over that index.

Apply both to `year` (a full year of business days, in the stub) as `fire_dates` and `n_fires`.

In [ ]:
year = pd.bdate_range("2023-01-01", "2023-12-31")


def month_start_days(dates):
    # TODO: the first available date within each calendar month
    ...


def rebalance_count(dates):
    # TODO: how many times does month_start fire?
    ...


fire_dates = month_start_days(year)
n_fires = rebalance_count(year)

print(f"{n_fires} rebalances")
for d in fire_dates[:4]:
    print("  ", d.date(), d.day_name())

In [ ]:
assert n_fires == 12, f"a full year has 12 monthly rebalances, got {n_fires}"
assert len(fire_dates) == 12, "one firing date per month"
assert all(isinstance(d, pd.Timestamp) for d in fire_dates), "return Timestamps"

# 1 Jan 2023 was a Sunday, so the January rebalance fires on Monday the 2nd.
assert fire_dates[0] == pd.Timestamp("2023-01-02"), (
    f"got {fire_dates[0].date()}. 2023-01-01 was a Sunday - month_start fires on the "
    "first day the market is OPEN, not the calendar 1st.")
# 1 Apr 2023 was a Saturday -> Monday the 3rd.
assert fire_dates[3] == pd.Timestamp("2023-04-03"),     f"April 1st 2023 was a Saturday; expected 2023-04-03, got {fire_dates[3].date()}"
assert fire_dates[1] == pd.Timestamp("2023-02-01"), "1 Feb 2023 was a Wednesday - no shift"
assert all(d.weekday() < 5 for d in fire_dates), "a schedule never fires on a weekend"
assert fire_dates == sorted(fire_dates), "dates must come back in order"

shifted = sum(1 for d in fire_dates if d.day != 1)
print(f"✅ Correct!  12 rebalances, and {shifted} of them did NOT land on the 1st.",
      "Never assume a schedule fires on the calendar date you had in mind.")

## Capstone 1 — Moving Average Crossover (Pillars 1–4)

The classic algo-trading starter: go long when a fast SMA crosses above a slow SMA ("golden cross"), exit when it crosses back below ("death cross"). Each pillar is labelled in the comments so you can trace how the five building blocks fit together.

**How to run it:** copy the entire class into a new QuantConnect algorithm project and click **Backtest**.

Read through the code top to bottom before running. Notice how the pillar labels map directly to the five sections you just studied.

In [ ]:
class MovingAverageCrossover(QCAlgorithm):
    # MA crossover on SPY using Pillars 1 through 4.

    # ── Pillar 1: Initialize ─────────────────────────────────────────────────
    def initialize(self):
        self.set_start_date(2020, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        self.symbol = self.add_equity("SPY", Resolution.DAILY).symbol

        # ── Pillar 3: Indicators ─────────────────────────────────────────────
        self.fast = self.sma(self.symbol, 20, Resolution.DAILY)   # 20-day SMA
        self.slow = self.sma(self.symbol, 50, Resolution.DAILY)   # 50-day SMA
        self.set_warm_up(50)   # pre-load 50 bars so both SMAs are ready on day 1

    # ── Pillar 2: Data ───────────────────────────────────────────────────────
    def on_data(self, data: Slice):
        if self.symbol not in data:
            return

        # ── Pillar 3: Check indicators ────────────────────────────────────────
        if not self.fast.is_ready or not self.slow.is_ready:
            return

        fast_val = self.fast.current.value
        slow_val = self.slow.current.value
        price    = data[self.symbol].close

        # ── Pillar 4: Portfolio & Orders ──────────────────────────────────────
        invested = self.portfolio[self.symbol].invested

        # Golden cross: fast SMA moves above slow SMA -> enter long
        if fast_val > slow_val and not invested:
            self.set_holdings(self.symbol, 1.0)
            self.log(f"BUY  SPY @ {price:.2f}  (SMA20={fast_val:.2f} > SMA50={slow_val:.2f})")

        # Death cross: fast SMA drops below slow SMA -> exit
        elif fast_val < slow_val and invested:
            self.liquidate(self.symbol)
            self.log(f"SELL SPY @ {price:.2f}  (SMA20={fast_val:.2f} < SMA50={slow_val:.2f})")

## Capstone 2 — Monthly Cross-Sectional Momentum (All 5 Pillars)

Cross-sectional momentum ranks stocks by recent return and bets the top performers continue to lead. This version:
- Builds a **dynamic universe** of liquid US stocks (Pillar 5 — Universe)
- Scores each stock by **12-minus-1-month return** (12-month total minus the most recent month, which reduces short-term reversal noise)
- **Rebalances monthly**, equal-weighting the top 20 (Pillar 5 — Scheduling)
- Uses Pillars 1–4 for setup, data history, and order execution

**How to run it:** paste the class into a QuantConnect project. Expect the backtest to take longer than Capstone 1 — it computes returns across ~200 stocks each month.

Read `initialize` first to understand the full setup, then trace how `rebalance` calls `self.history()` to compute the momentum signal. Notice that `on_data` is empty — all trading happens inside `rebalance`.

In [ ]:
class MonthlyMomentum(QCAlgorithm):
    # Cross-sectional momentum: top-20 US stocks by 12-1m return, rebalanced monthly.

    # ── Pillar 1: Initialize ─────────────────────────────────────────────────
    def initialize(self):
        self.set_start_date(2019, 1, 1)
        self.set_end_date(2023, 12, 31)
        self.set_cash(100_000)

        # SPY as a market-hours anchor for scheduling; may appear in universe too
        self.spy = self.add_equity("SPY", Resolution.DAILY).symbol

        # ── Pillar 5: Universe ────────────────────────────────────────────────
        self.universe_symbols = []
        self.add_universe(self.coarse_filter)

        # ── Pillar 5: Schedule ────────────────────────────────────────────────
        self.schedule.on(
            self.date_rules.month_start(),
            self.time_rules.after_market_open(self.spy, 30),
            self.rebalance
        )

    # ── Pillar 5: Universe filter ────────────────────────────────────────────
    def coarse_filter(self, coarse):
        # Liquid stocks: has fundamental data, price > $5, dollar volume > $1M/day
        eligible = [x for x in coarse
                    if x.has_fundamental_data
                    and x.price > 5
                    and x.dollar_volume > 1_000_000]
        # Pool of top 200 by dollar volume -- large enough to rank for momentum
        return [x.symbol for x in sorted(eligible,
                                         key=lambda x: x.dollar_volume,
                                         reverse=True)[:200]]

    # ── Pillar 5: Universe events ─────────────────────────────────────────────
    def on_securities_changed(self, changes):
        for security in changes.added_securities:
            if security.symbol not in self.universe_symbols:
                self.universe_symbols.append(security.symbol)
        for security in changes.removed_securities:
            self.liquidate(security.symbol)
            if security.symbol in self.universe_symbols:
                self.universe_symbols.remove(security.symbol)

    # ── Pillar 2 + 4: Scheduled rebalance ────────────────────────────────────
    def rebalance(self):
        if len(self.universe_symbols) < 20:
            return   # not enough stocks to rank yet

        # history() returns a pandas DataFrame with a (symbol, time) multi-index
        history = self.history(self.universe_symbols, 252, Resolution.DAILY)

        scores = {}
        for symbol in self.universe_symbols:
            try:
                closes = history.loc[symbol]["close"]
                if len(closes) < 22:
                    continue
                ret_12m = closes.iloc[-1] / closes.iloc[0]   - 1   # 12-month return
                ret_1m  = closes.iloc[-1] / closes.iloc[-22] - 1   # last-month return
                scores[symbol] = ret_12m - ret_1m                   # 12-1 momentum score
            except Exception:
                continue

        if not scores:
            return

        # ── Pillar 4: Orders ──────────────────────────────────────────────────
        top20  = sorted(scores, key=scores.get, reverse=True)[:20]
        weight = 1.0 / len(top20)

        # Exit positions no longer in the top-20
        for symbol in self.universe_symbols:
            if symbol not in top20 and self.portfolio[symbol].invested:
                self.liquidate(symbol)

        # Enter or reweight top-20 at equal weight
        for symbol in top20:
            self.set_holdings(symbol, weight)

        self.log(f"Rebalanced: top {len(top20)} momentum stocks at {weight:.1%} each")

    def on_data(self, data: Slice):
        pass   # all trading happens in rebalance()

## Cheat sheet

| Pillar | Method / Pattern | What it does |
|---|---|---|
| **1 — Initialize** | `self.set_start_date(y, m, d)` | Backtest start date |
| | `self.set_end_date(y, m, d)` | Backtest end date |
| | `self.set_cash(n)` | Starting capital in USD |
| | `self.add_equity("SPY", Resolution.DAILY).symbol` | Subscribe to daily data |
| | `self.set_warm_up(n)` | Pre-load n bars of history |
| | `self.set_benchmark("QQQ")` | Custom benchmark |
| **2 — Data** | `def on_data(self, data: Slice):` | Called every bar |
| | `if self.symbol not in data: return` | Guard against missing data |
| | `data[symbol].close / .open / .high / .low / .volume` | Price fields |
| | `self.log(message)` | Write to backtest log |
| **3 — Indicators** | `self.sma(symbol, period, Resolution.DAILY)` | Simple Moving Average |
| | `self.ema(symbol, period)` | Exponential Moving Average |
| | `self.rsi(symbol, period)` | RSI (0–100) |
| | `self.bb(symbol, period, k)` | Bollinger Bands |
| | `.is_ready` / `.current.value` | Check readiness / read value |
| **4 — Portfolio** | `self.portfolio[symbol].invested` | Currently holding? |
| | `self.set_holdings(symbol, weight)` | Target weight (0 = flat, 1.0 = fully long) |
| | `self.liquidate(symbol)` | Exit position |
| | `self.market_order(symbol, qty)` | Buy/sell exact shares |
| **5 — Universe** | `self.add_universe(filter_fn)` | Dynamic stock selection |
| | `def on_securities_changed(self, changes):` | Handle entries/exits |
| | `self.schedule.on(date_rule, time_rule, fn)` | Recurring function |
| | `self.date_rules.month_start()` | First day of each month |
| | `self.time_rules.after_market_open(symbol, n)` | n minutes after open |

## Stretch goals

Bring these to the next meeting:

1. **Vary Capstone 1 parameters** — change the fast/slow periods (try 10/30 or 5/20). Does the crossover become more or less profitable? More or fewer trades per year?
2. **Add a volatility filter to Capstone 1** — only enter when ATR is below a threshold. Use `self.atr(self.symbol, 14)` and check `.current.value` against a fixed level.
3. **Modify Capstone 2's lookback** — try 6-month momentum (126 bars) instead of 12-month (252 bars). Does shorter-term momentum perform better or worse?
4. **Add a stop-loss to Capstone 1** — if the position falls more than 5% from entry, liquidate. Track the entry price on `self` and compare it to the current close in `on_data`.
5. **Combine Pillars 4 and 5** — use Capstone 2's universe but add an RSI filter in `rebalance`: only include top-20 stocks where RSI < 60 (avoid overbought entries).

## What's next

**Module 4 — The Research Environment & Working with Financial Data** picks up where Module 2's pandas skills left off and brings them into QuantConnect's interactive Research Environment. You'll use `QuantBook` to pull real market data, compute return series, run rolling statistics, and plot correlation matrices — all live in a Jupyter-style notebook on QuantConnect's servers.

**Official docs:**
- [Initialization](https://www.quantconnect.com/docs/v2/writing-algorithms/initialization)
- [Handling Data / Slice](https://www.quantconnect.com/docs/v2/writing-algorithms/securities/handling-data)
- [Indicators](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/supported-indicators)
- [Portfolio & Orders](https://www.quantconnect.com/docs/v2/writing-algorithms/trading-and-orders/order-types/market-orders)
- [Universe Selection](https://www.quantconnect.com/docs/v2/writing-algorithms/universes/equity/coarse-universe-selection)
- [Scheduling](https://www.quantconnect.com/docs/v2/writing-algorithms/scheduled-events)
- [PEP8 snake_case migration](https://www.quantconnect.com/announcements/16830/pep8-python-api-migration/)

*MAT Education · QuantConnect Core · Module 3.*